# 08 Live HF spectator

Run a **multi-agent simulation** with **real Hugging Face Inference** models. Each agent can use a **different Hub instruct model** (rotated or shuffled from your pool). Events appear **as they happen** via `SimulationEngine.run(..., on_event=...)`.

**Requirements:** `HF_TOKEN` (`.env` at repo root, **`hf_token.txt`** first line, or your shell env), `pip install -e '.[notebooks]'`, and models your account can call (some gated repos need license acceptance on the Hub).

> Default base model in the library is **Llama 3.1 8B Instruct**; this notebook uses the **curated small instruct pool** from `agent_rpg.model_catalog`.

> **Import issues after `git pull`?** Use **Kernel → Restart**, then run cells from the top so Jupyter drops a stale `agent_rpg` from memory.

> **HTTP 402 from the Hub?** That is billing / included Inference credits exhausted — see [HF billing settings](https://huggingface.co/settings/billing), or use mock/local backends for free development runs.

In [ ]:
from pathlib import Path
import sys
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src = base / "src"
    if (src / "agent_rpg").is_dir():
        if str(src) not in sys.path:
            sys.path.insert(0, str(src))
        break
from agent_rpg.repo_root import find_repo_root
ROOT = find_repo_root()
print("Repository root:", ROOT)


## 1 — Imports and environment

In [ ]:
import os

try:
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env")
except ImportError:
    pass

if not os.environ.get("HF_TOKEN"):
    for _name in ("hf_token.txt", "HF_TOKEN.txt"):
        _p = ROOT / _name
        if _p.is_file():
            _line = (_p.read_text(encoding="utf-8").strip().splitlines() or [""])[0].strip()
            if _line:
                os.environ["HF_TOKEN"] = _line
            break

if not os.environ.get("HF_TOKEN"):
    print("WARNING: HF_TOKEN not set — Inference calls will fail.")
    print("Fix: export HF_TOKEN=hf_...  OR  create .env with HF_TOKEN=...  OR  put the token on the first line of hf_token.txt (gitignored).")
else:
    print("HF_TOKEN is set (value hidden).")

## 2 — Choose models (ipywidgets)
- **Pool**: pick one or more Hub ids from the curated list (hold Cmd/Ctrl for multi-select).  
- **Strategy**: how to assign models across agents (`rotate` / `shuffle` / `random`).  
- **Stream tokens**: print model output token-by-token as it arrives (noisy but vivid).

In [ ]:
import ipywidgets as W
from IPython.display import display

from agent_rpg.model_catalog import SMALL_INSTRUCT_MODELS

labels = [m["label"] for m in SMALL_INSTRUCT_MODELS]
ids = [m["id"] for m in SMALL_INSTRUCT_MODELS]
label_to_id = dict(zip(labels, ids))

pool_sel = W.SelectMultiple(
    options=[(lbl, lbl) for lbl in labels],
    value=(labels[0],),
    rows=min(8, len(labels)),
    description="Model pool",
)
strat = W.Dropdown(
    options=[("rotate", "rotate"), ("shuffle", "shuffle"), ("random", "random")],
    value="rotate",
    description="Strategy",
)
rounds = W.IntSlider(value=2, min=1, max=6, description="Rounds")
n_agents = W.IntSlider(value=3, min=2, max=6, description="Agents")
seed = W.IntText(value=42, description="Seed")
stream_tokens = W.Checkbox(value=False, description="Stream tokens")
thoughts = W.Checkbox(value=False, description="Thought phase")

ui = W.VBox([pool_sel, strat, rounds, n_agents, seed, stream_tokens, thoughts])
display(ui)

## 3 — Build scenario and assign per-agent models

In [ ]:
from agent_rpg import SimulationEngine, build_random_scenario
from agent_rpg.backends.hf_inference import HuggingFaceInferenceBackend
from agent_rpg.multi_model import assign_models_to_agents, set_router_model_if_reactive

chosen_labels = tuple(pool_sel.value) if pool_sel.value else (labels[0],)
model_pool = [label_to_id[l] for l in chosen_labels]

scenario = build_random_scenario(
    seed=int(seed.value),
    num_agents=int(n_agents.value),
    max_rounds=int(rounds.value),
    model_id=model_pool[0],
)
assign_models_to_agents(scenario, model_pool, strategy=strat.value, rng=__import__("random").Random(int(seed.value)))
set_router_model_if_reactive(scenario, model_pool[0])
scenario.orchestration.enable_thought_phase = bool(thoughts.value)
for ag in scenario.agents:
    ag.max_new_tokens = min(ag.max_new_tokens, 320)
    ag.temperature = min(ag.temperature, 0.85)

print("World:", scenario.world.title)
for a in scenario.agents:
    print(f"  {a.display_name}: {a.model_id}")

## 4 — Spectate the run (live events)
`on_event` fires after each log line is written — **messages** and **thoughts** are shown as Markdown. Optional **token streaming** prints raw deltas under the same cell (stderr-style).

In [ ]:
from IPython.display import Markdown, display


def on_event(ev):
    if ev.event_type == "world_event":
        display(Markdown(f"**World** (round {ev.round}): {ev.payload.get('description', '')}"))
    elif ev.event_type == "thought":
        aid = ev.agent_id or "?"
        display(Markdown(f"_Thought — **{aid}**_: {ev.payload.get('text', '')}"))
    elif ev.event_type == "message":
        aid = ev.agent_id or "?"
        say = ev.payload.get("text", "")
        display(Markdown(f"**{aid}**: {say}"))
    elif ev.event_type == "router":
        display(Markdown(f"_Router_: next speaker hint `{ev.payload.get('chosen')!s}`"))
    elif ev.event_type == "error":
        display(Markdown(f"⚠ **error**: `{ev.payload}`"))
    elif ev.event_type == "system":
        display(Markdown(f"_System_: `{ev.payload}`"))

backend = HuggingFaceInferenceBackend()
llm_extras: dict = {}
if stream_tokens.value:

    def chunk_cb(t: str) -> None:
        print(t, end="", flush=True)

    llm_extras = {"stream": True, "chunk_callback": chunk_cb}

run_id = f"live_spec_{int(seed.value)}"
print("--- simulation start ---\n")
out = SimulationEngine(scenario).run(
    backend,
    output_dir=ROOT / "runs",
    run_id=run_id,
    seed=int(seed.value),
    on_event=on_event,
    llm_extras=llm_extras,
)
print("\n--- done ---\nJSONL:", out / "events.jsonl")

## 5 — Optional: quick report

In [ ]:
from agent_rpg import ReportBuilder

rb = ReportBuilder.from_jsonl(out / "events.jsonl")
d = rb.to_dict(scenario=scenario)
print("Messages:", d["summary"]["message_count"], "Errors:", d["summary"]["error_count"])
print("Gini:", round(d["social_dynamics"]["gini_turns"], 4))